# Lesson 4: ReRank

In [76]:
import warnings

# Disable all warnings
warnings.filterwarnings("ignore")

import os
import weaviate
import cohere
from dotenv import load_dotenv
from utils import dense_retrieval
from utils import print_result
from utils import keyword_search

load_dotenv()
os.environ['WEAVIATE_API_URL'] = 'https://yvi3vi7ytcqjshbpzpeta.c0.us-west3.gcp.weaviate.cloud/'
os.environ['WEAVIATE_API_KEY'] = 'Qlh3QzlUa0M3ZEtKelprcV9VSTBlRHcwRU1YZUVrOU5TZ3dxMENGWXMvWHMreDBDRHlJYWpjd3lpQWEwPV92MjAw'
co = cohere.Client(os.environ['COHERE_API_KEY'])
auth_config = weaviate.auth.AuthApiKey(
    api_key=os.getenv("WEAVIATE_API_KEY")
)

client = weaviate.Client(
    url=os.getenv("WEAVIATE_API_URL"),
    auth_client_secret=auth_config,
    additional_headers={
        "X-Cohere-Api-Key": os.getenv("COHERE_API_KEY"),
    }
)

# Dense Retrieval

In [77]:
# --- Force-seed a Canada doc so the "capital of Canada" query matches ---
with client.batch as b:
    b.add_data_object({
        "title": "Canada",
        "url": "https://en.wikipedia.org/wiki/Canada",
        "text": "Ottawa is the capital city of Canada. Toronto is the most populous city.",
        "views": 123,
        "lang": "en",
    }, class_name="Articles")

# sanity: show new count
agg = client.query.aggregate("Articles").with_meta_count().do()
print("Articles count:", agg["data"]["Aggregate"]["Articles"][0]["meta"]["count"])

Articles count: 25


In [78]:
query = "What is the capital of Canada?"

In [79]:
# ---- INSPECT + FIX BM25 SEARCH (run after client is created) ----
import json, cohere

# 1) Peek at schema & a few docs to confirm field names/content
schema = client.schema.get()
print("Classes:", [c["class"] for c in schema.get("classes", [])])
art = next((c for c in schema.get("classes", []) if c.get("class")=="Articles"), None)
print("Articles properties:", [p["name"] for p in (art or {}).get("properties", [])])

sample = (client.query
          .get("Articles", ["_additional {id}", "title", "lang", "url", "views", "text"])
          .with_limit(5)
          .do()
         ).get("data",{}).get("Get",{}).get("Articles",[])
print("Sample docs (first 5):")
for i, s in enumerate(sample, 1):
    tl = (s.get("text") or "")[:80].replace("\n"," ")
    print(f"{i}. title={s.get('title')} | lang={s.get('lang')} | text[:80]={tl}")

# 2) Count objects so we know we're hitting the right class
agg = client.query.aggregate("Articles").with_meta_count().do()
count = agg["data"]["Aggregate"]["Articles"][0]["meta"]["count"]
print("Articles count:", count)

# 3) If nothing looks like natural language in 'text', seed a small doc so BM25 can match
def seed_if_needed():
    # Try a pure GET (no BM25) to see if any doc has non-empty 'text'
    some = (client.query.get("Articles", ["text"]).with_limit(3).do()
           ).get("data",{}).get("Get",{}).get("Articles",[])
    has_text = any((d.get("text") or "").strip() for d in some)
    if has_text:
        print("At least one doc has a non-empty 'text' field.")
        return
    print("No non-empty 'text' fields detected in sample; seeding one doc...")
    with client.batch as b:
        b.add_data_object({
            "title": "Canada",
            "url": "https://en.wikipedia.org/wiki/Canada",
            "text": "Ottawa is the capital city of Canada. Toronto is the most populous city.",
            "views": 123,
            "lang": "en",
        }, class_name="Articles")
    print("Seeded one doc.")

seed_if_needed()

# 4) Force BM25 to search specific fields (title + text) and print results
query = "What is the capital of Canada?"
props_to_return = ["title","url","lang","views","text"]  # what to return
props_to_search = ["title","text"]                       # where BM25 should search

bm25 = (client.query.get("Articles", props_to_return)
        .with_bm25(query=query, properties=props_to_search)
        .with_limit(50)
        .do()
       ).get("data",{}).get("Get",{}).get("Articles",[])
print("BM25 hits:", len(bm25))
for i, r in enumerate(bm25[:5], 1):
    print(f"{i}. {r.get('title')} | lang={r.get('lang')}")
    print((r.get('text') or "")[:200], "\n")

# 5) If still no hits, try a different query and print the first doc to see content shape
if not bm25:
    print("\nNo hits for that query. Trying a generic term and showing one full doc:")
    probe = (client.query.get("Articles", props_to_return)
             .with_bm25(query="the", properties=props_to_search)
             .with_limit(5)
             .do()
            ).get("data",{}).get("Get",{}).get("Articles",[])
    print("Probe hits:", len(probe))
    first = (client.query.get("Articles", props_to_return).with_limit(1).do()
            ).get("data",{}).get("Get",{}).get("Articles",[])
    print("First stored doc:", json.dumps(first[0] if first else {}, indent=2)[:1000])

# 6) Rerank (only if we have texts)
def rerank_responses(query, responses, num_responses=10):
    if not responses:
        return []
    co = cohere.Client(os.environ["COHERE_API_KEY"])
    for model in [os.getenv("COHERE_RERANK_MODEL") or "rerank-v3.5",
                  "rerank-multilingual-v3.0", "rerank-english-v3.0"]:
        try:
            return co.rerank(
                query=query,
                documents=responses,
                top_n=min(num_responses, len(responses)),
                model=model,
                return_documents=True,
            )
        except cohere.errors.NotFoundError:
            continue
    raise RuntimeError("No accessible Cohere rerank model.")

texts = [r.get("text","") for r in bm25 if r.get("text")]
print("Texts available for rerank:", len(texts))
if texts:
    rr = rerank_responses(query, texts, num_responses=5)
    results = getattr(rr, "results", rr)
    print("Top reranked:")
    for i, item in enumerate(results, 1):
        score = getattr(item, "relevance_score", getattr(item, "score", None))
        print(f"#{i} score={score}")
        print((getattr(item, "document", "") or "")[:200], "\n")


Classes: ['Articles']
Articles properties: ['title', 'url', 'text', 'views', 'lang']
Sample docs (first 5):
1. title=Final de la Copa Mundial | lang=es | text[:80]=La final de la Copa Mundial de la FIFA es uno de los eventos televisados más vis
2. title=نهائي كأس العالم | lang=ar | text[:80]=نهائي كأس العالم FIFA يعد من أكثر الأحداث التلفزيونية مشاهدةً في العالم.
3. title=FIFA World Cup Final | lang=en | text[:80]=The FIFA World Cup final is one of the most watched televised events worldwide.
4. title=FIFA-Weltmeisterschaftsfinale | lang=de | text[:80]=Das Finale der FIFA-Weltmeisterschaft ist eines der meistgesehenen Fernsehereign
5. title=FIFA विश्व कप फाइनल | lang=hii | text[:80]=FIFA विश्व कप का फाइनल दुनिया के सबसे अधिक देखे जाने वाले टेलीविज़न आयोजनों में 
Articles count: 25
At least one doc has a non-empty 'text' field.
BM25 hits: 2
1. Canada | lang=en
Ottawa is the capital city of Canada. Toronto is the most populous city. 

2. Canada | lang=en
Ottawa is the capital city of Can

TypeError: 'RerankResponseResultsItemDocument' object is not subscriptable

In [80]:
#dense_retrieval_results = dense_retrieval(query, client
# BM25 keyword search
query = "What is the capital of Canada?"
bm25_results = keyword_search(
    query,
    client,
    properties=["text", "title", "url", "views", "lang", "_additional {distance}"],
    num_results=50
)

# ReRank with Cohere
texts = [r.get("text","") for r in bm25_results if r.get("text")]
reranked = rerank_responses(query, texts, num_responses=10)

# Pretty-print top reranked docs
for i, item in enumerate(reranked):
    doc = item.document
    print(f"#{i+1} | score={item.relevance_score:.3f}")
    print(doc[:500], "\n")


AttributeError: 'tuple' object has no attribute 'document'

In [81]:
print_result(reranked)

item 0


AttributeError: 'tuple' object has no attribute 'keys'

# Improving Keyword Search with ReRank

In [82]:
query_1 = "What is the capital of Canada?"

In [83]:
query_1 = "What is the capital of Canada?"
results = keyword_search(query_1,
                         client,
                         properties=["text", "title", "url", "views", "lang", "_additional {distance}"],
                         num_results=3
                        )

for i, result in enumerate(results):
    print(f"i:{i}")
    print(result.get('title'))
    print(result.get('text'))

i:0
Canada
Ottawa is the capital city of Canada. Toronto is the most populous city.
i:1
Canada
Ottawa is the capital city of Canada. Toronto is the most populous city.


In [84]:
query_1 = "What is the capital of Canada?"
results = keyword_search(query_1,
                         client,
                         properties=["text", "title", "url", "views", "lang", "_additional {distance}"],
                         num_results=500
                        )

for i, result in enumerate(results):
    print(f"i:{i}")
    print(result.get('title'))


i:0
Canada
i:1
Canada


In [85]:
# import cohere


# def rerank_responses(query, responses, num_responses=10):
#     co = cohere.Client(os.environ['COHERE_API_KEY'])
#     reranked_responses = co.rerank(
#         query=query,
#         documents=responses,
#         top_n=num_responses,  # controls how many reranked docs to return
#         model="rerank-v3.5"
#     )
#     return reranked_responses

In [86]:
texts = [result.get('text') for result in results]
reranked_text = rerank_responses(query_1, texts)
print(reranked_text)

id='836b7c3b-3541-416a-89c6-2103ef42dbde' results=[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Ottawa is the capital city of Canada. Toronto is the most populous city.'), index=0, relevance_score=0.86954874), RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Ottawa is the capital city of Canada. Toronto is the most populous city.'), index=1, relevance_score=0.86954874)] meta=ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(input_tokens=None, output_tokens=None, search_units=1, classifications=None), tokens=None, warnings=None)


In [87]:
for i, rerank_result in enumerate(reranked_text):
    print(f"i:{i}")
    print(f"{rerank_result}")
    print()

i:0
('id', '836b7c3b-3541-416a-89c6-2103ef42dbde')

i:1
('results', [RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Ottawa is the capital city of Canada. Toronto is the most populous city.'), index=0, relevance_score=0.86954874), RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Ottawa is the capital city of Canada. Toronto is the most populous city.'), index=1, relevance_score=0.86954874)])

i:2
('meta', ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(input_tokens=None, output_tokens=None, search_units=1, classifications=None), tokens=None, warnings=None))



# Improving Dense Retrieval with ReRank

In [88]:
query_2 = "Who is the tallest person in history?"
#results = dense_retrieval(query_2,client)

In [89]:
for i, result in enumerate(results):
    print(f"i:{i}")
    print(result.get('title'))
    print(result.get('text'))
    print()

i:0
Canada
Ottawa is the capital city of Canada. Toronto is the most populous city.

i:1
Canada
Ottawa is the capital city of Canada. Toronto is the most populous city.



In [90]:
texts = [result.get('text') for result in results]
reranked_text = rerank_responses(query_2, texts)

In [91]:
for i, rerank_result in enumerate(reranked_text):
    print(f"i:{i}")
    print(f"{rerank_result}")
    print()

i:0
('id', 'ef325b85-45f7-495b-ba39-5d8645adadcb')

i:1
('results', [RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Ottawa is the capital city of Canada. Toronto is the most populous city.'), index=0, relevance_score=0.03749929), RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Ottawa is the capital city of Canada. Toronto is the most populous city.'), index=1, relevance_score=0.03749929)])

i:2
('meta', ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(input_tokens=None, output_tokens=None, search_units=1, classifications=None), tokens=None, warnings=None))

